# Simple AI Workflow
---

## Introduction
> Here we will create a simple `classifier` trained from `FashionMNIST dataset`

## Data
---
> Dealing with data involved two primitives: `Datasets` and `DataLoaders`
> Domain specific datasets can be retrived from difference packages `torchvision`, `torchaudio`, `torchtext`, etc.
> Though it is personally better to create your own datasets. Use existing ones if there are any.

In [1]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader

### Datasets
> Download existing datasets from FashionMNIST

In [2]:
# here we use transform already to transform data to torch data type and perform scaling
from torchvision.transforms import v2

In [3]:
training_data = datasets.FashionMNIST(
    root='./data/',
    train=True,
    download=True,
    transform=v2.Compose(transforms=[v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
    )
test_data = datasets.FashionMNIST(
    root='./data/',
    train=False,
    download=True,
    transform=v2.Compose(transforms=[v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
    )

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.82MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.9MB/s]


### DataLoaders
> Initialize `DataLoaders` which is an iterable of the Dataset that supports `automatic batching`, `sampling`, `shuffling` and `multi-process data loading`.

In [4]:
batch_size=64
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=True)

In [18]:
# Inspect dataset sample dimensions
# Usually data loaders loads Sample feature and labels during a single iteration

for batch, (X,y) in enumerate(train_dataloader):
  print(f"Batch: {batch}")
  print(f"Batch Data Shape: {X.shape}") # BxCxHxW
  print(f"Batch Label Shape: {y.shape}")# Bx1 (single vector for lables, maybe need to one-hot-encode to alight with neural network output)
  break

Batch: 0
Batch Data Shape: torch.Size([64, 1, 28, 28])
Batch Label Shape: torch.Size([64])


## Model
> Here we construct a simple classifier with `3 layer FFN` stack with `ReLU` activations.
> We use available acceleator

We can use `torch.nn.Sequential` to stack FFN layers.


In [17]:
from torch import nn
torch.cuda.is_available()
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available else 'cpu'
print(f"Using device {device}")

Using device cuda


### Model Definition

In [53]:
class NNClassifier(nn.Module):
  def __init__(self, c=1, h=28, w=28, o=10):
    super().__init__()

    self.flatten = nn.Flatten() #  CxHxW-> CHW
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(c*h*w, 512), # CHW->512
        nn.ReLU(),
        nn.Linear(512,512), # 512 -> 512
        nn.ReLU(),
        nn.Linear(512,10) # 512->10
        # no need for final ReLU, softmax is baked onl CE loss
    )
  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

## Model Training

Use `CE` loss function for multi-classification problem.
For this scenario we use `SGD` with `lr=1e-3`

In [74]:
# Loss and optimizers

classifier = NNClassifier(1,28,28,10).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(classifier.parameters(), lr=1e-3)

### Training Loop

In [75]:
epochs = 5
def train(dataloader, model, loss_fn, optimizer):
  model.train() # model in train mode
  # Training Loop
  for epoch in range(1,epochs+1):
    for batch, (X,y) in enumerate(dataloader):
      # Compute loss
      X,y = X.to(device), y.to(device)
      pred = model(X)
      loss = loss_fn(pred, y)

      # Backpropagate Loss
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      # Compute batch loss every 100'th batch
      if batch % 100 == 0: #
        print(f"epoch: {epoch}\t batch: {batch}\t loss: {loss.item()}")

In [76]:
train(dataloader=train_dataloader, model=classifier, loss_fn=loss_fn, optimizer=optimizer)

epoch: 1	 batch: 0	 loss: 2.317125082015991
epoch: 1	 batch: 100	 loss: 2.287785530090332
epoch: 1	 batch: 200	 loss: 2.268742322921753
epoch: 1	 batch: 300	 loss: 2.259065628051758
epoch: 1	 batch: 400	 loss: 2.245663642883301
epoch: 1	 batch: 500	 loss: 2.232189178466797
epoch: 1	 batch: 600	 loss: 2.2188546657562256
epoch: 1	 batch: 700	 loss: 2.212221145629883
epoch: 1	 batch: 800	 loss: 2.1923277378082275
epoch: 1	 batch: 900	 loss: 2.167357921600342
epoch: 2	 batch: 0	 loss: 2.1524412631988525
epoch: 2	 batch: 100	 loss: 2.129837989807129
epoch: 2	 batch: 200	 loss: 2.122419834136963
epoch: 2	 batch: 300	 loss: 2.097670078277588
epoch: 2	 batch: 400	 loss: 2.064580202102661
epoch: 2	 batch: 500	 loss: 2.0340416431427
epoch: 2	 batch: 600	 loss: 2.01448392868042
epoch: 2	 batch: 700	 loss: 1.9636869430541992
epoch: 2	 batch: 800	 loss: 1.9690473079681396
epoch: 2	 batch: 900	 loss: 1.8935219049453735
epoch: 3	 batch: 0	 loss: 1.8837223052978516
epoch: 3	 batch: 100	 loss: 1.883947

### Test

In [77]:
def test(dataloader, model, loss_fn):
  model.eval() # model in eval mode
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0,0
  with torch.no_grad():
    for X,y in dataloader:
      X,y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_fn(pred, y)
      # get the prediction argmax at 2nd dimension, then sum along batch
      correct += (pred.argmax(1)==y).type(torch.float32).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Test loss: {test_loss} with accuracy {correct}")


In [78]:
test(test_dataloader, classifier, loss_fn)

Test loss: 1.1156412363052368 with accuracy 0.6471
